# AXI test 14: what the GPU spends on a job when many jobs share it (the serving side)
**Why this test exists.** The 87 percent fewer tokens was counted for one job in a plain loop. A company that runs the model has a prefix cache, so a kept chat is cheaper there. This test measures what is left: GPU time and cache memory, with many jobs on one GPU, **exact-state against a kept chat that has the cache on**.

Model: **Qwen2.5-7B-Instruct, 4-bit AWQ**, engine **vLLM**, cache fixed at **80,000 tokens**. Cells: `S` (exact state each step) and `KC` (whole chat kept) with 8, 32 and 64 jobs side by side; `KN` (chat kept, cache emptied each round) with 8 jobs; `KR` (like `KC`, random order) with 32 jobs.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **How long it takes (measured on the first run, 17 and 18 Oct 2026, a free T4):** `S` cells take under 4 minutes each, `KC` at 8 jobs 2.5 minutes, `KN` at 8 jobs 27 minutes, `KC` at 32 jobs 82 minutes. `KC` at 64 jobs did not finish there (the free GPU time ran out inside it); my guess for it is **three hours or more**, which is a guess. Every finished cell is saved to your Drive, so if Colab stops you can reconnect, run Cells 2 to 5 again (with Cell 4b), and the run cells (Cell 7 onwards) skip what is finished. **A cell that is cut off in the middle is not saved.**

**If you are running only the missing cell** (`KC` at 64 jobs): run Cells 1 to 6 (Cell 4b included), skip Cells 7, 8 and 10, and run Cell 9; it skips the `S` cell at 64 jobs if its file is in your Drive and runs `KC` at 64 jobs on **vLLM 0.31.0**, the version of the first run (a cell from another version is refused when the cells are put together). When it is done, the file to send me is `axi_provider_KC_K64.json` from your Drive folder, as before; you do not need Cell 11 in that case, I put the cells together myself.

Predictions and reading rules were written before this run in `docs/PROVIDER_SIDE_TEST.md`. **Do not paste any keys anywhere.**

### Cell 1: check the GPU (it must say Tesla T4, or another GPU with at least 15 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 57 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side.py"), "the code is not on GitHub yet: finish the update 57 commands (git push) first"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 5 again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 5 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 5

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: load the model and the engine (downloads about 5.5 GB once; about 5 to 10 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first)

In [ ]:
from axi.experiments import provider_side as P
eng = P.VLLMEngine()
info = eng.info(); print(json.dumps(info, indent=1))
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(arm, K):
    """Runs one cell (arm, K) unless its file is already there; saves it and copies it to Drive."""
    name = f"axi_provider_{arm}_K{K}.json"
    print(f"=== {arm} K={K}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    t0 = time.time()
    c = P.run_cell(eng, arm, K, progress=lambda i, a, k: print(f"   {a} K={k}  round {i}   {time.time() - t0:6.0f} s", end="\r"))
    P.save_cell(c, store.path(name)); store.persist(name)
    s = P.summarize_cell(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 2)} s/step, computed prompt tokens {f(s['computed_prefill'], 0)}, cached share {f(s['cached_share'] and s['cached_share'] * 100, 0)} %, exact steps {s['exact_share'] * 100:.0f} %")

### Cell 6: SAFETY CHECK (about 3 minutes). 3 jobs of 3 steps in each of S, KC, KN. It prints, for the first job, the prompt size and how many tokens the engine says came from its cache. If it says STOP, do not run the next cells: send me the printout

In [ ]:
smoke = {arm: P.run_cell(eng, arm, 3, steps=3) for arm in ("S", "KC", "KN")}
for arm, c in smoke.items():
    for x in c["sessions"][0]["log"]: print(f"{arm:2s} step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}")
ok_cached = all(x["n_cached"] is not None for c in smoke.values() for s in c["sessions"] for x in s["log"])
kc = smoke["KC"]["sessions"][0]["log"]
ok_work = ok_cached and kc[1]["n_cached"] > 0
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached else
      "STOP: the cache did nothing for the kept chat's second step. Send me the printout." if not ok_work else "OK: cached counts are reported and the cache works. You can go on.")

### Cell 7: the light load, 8 jobs side by side: S, KC and KN (about 30 minutes: S 1 minute, KC 2.5 minutes, KN 27 minutes; KN is the slow one). A counter shows the round

In [ ]:
for arm in ("S", "KC", "KN"): go(arm, 8)

### Cell 8: the middle load, 32 jobs: S and KC (about 1 hour 25 minutes: S 2 minutes, KC 82 minutes)

In [ ]:
for arm in ("S", "KC"): go(arm, 32)

### Cell 9: the heavy load, 64 jobs: S and KC (S about 4 minutes; KC is slow on purpose, that is what is being measured: my guess is three hours or more, not measured). If the free GPU time runs out inside a cell, that cell is not saved; the lines that need it will read 'not run'

In [ ]:
for arm in ("S", "KC"): go(arm, 64)

### Cell 10 (optional, about 1 hour 20 minutes, a guess): the same chat as KC with the jobs served in a random order each round, 32 jobs

In [ ]:
go("KR", 32)

### Cell 11: put the finished cells together, print the table and the 13 reading-rule lines, and download your file

In [ ]:
from google.colab import files
paths = sorted(glob.glob(store.path("axi_provider_*_K*.json")))
paths = [p for p in paths if not p.endswith("axi_provider_side.json")]
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = P.merge_files(paths, store.path("axi_provider_side.json")); store.persist("axi_provider_side.json")
S = P.summarize_all(m["cells"]); P.print_cells(S); P.print_verdict_provider(P.verdict_provider(S))
files.download(store.path("axi_provider_side.json"))

### Done
File downloaded: `axi_provider_side.json`. Upload it to the chat, with a screenshot of the printed table and of the 13 reading-rule lines, and the engine numbers printed by Cell 5 (vLLM version, GPU, cache size). Do not paste any keys anywhere.